## Section 1 — Setup: Clone repo, install deps, GPU validation


In [ ]:
!git clone https://github.com/RocketUp-Team/Sentiment-Analysis-Service.git || true
%cd Sentiment-Analysis-Service
!git checkout feature/ai-core
!pip uninstall -y torchvision torchaudio
!pip install -r requirements.txt
!nvidia-smi

## Section 2 — Credentials: Read Colab Secrets, pre-flight checks


In [ ]:
import os
from google.colab import userdata

os.environ['MLFLOW_TRACKING_URI'] = userdata.get('MLFLOW_TRACKING_URI')
os.environ['DAGSHUB_USER'] = userdata.get('DAGSHUB_USER')
os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['MODEL_VERSION'] = userdata.get('MODEL_VERSION')

# Add pre-flight checks here if needed
print("Secrets loaded.")


## Section 3 — Data Download


In [ ]:
!python -m src.data.downloader --task sarcasm
!python -m src.data.downloader --task sentiment


## Section 4 — Training


In [ ]:
import mlflow
from pathlib import Path
from src.scripts.run_finetuning import train

def _adapter_exists(task_name):
    return Path(f"models/adapters/{task_name}").exists()

mlflow.set_experiment("colab_pipeline_run")

with mlflow.start_run(run_name="full_pipeline") as parent_run:
    for task in ["sarcasm", "sentiment"]:
        if not _adapter_exists(task):
            with mlflow.start_run(run_name=f"train_{task}", nested=True):
                train(task)
        else:
            print(f"Skipping {task} training, adapter exists.")


## Section 5 — Evaluation


In [ ]:
from src.scripts.evaluate_finetuned import evaluate

with mlflow.start_run(run_id=parent_run.info.run_id):
    metrics_sarcasm = evaluate("sarcasm")
    mlflow.log_metric("sarcasm_overall_f1", metrics_sarcasm["overall_f1"])
    
    metrics_sentiment = evaluate("sentiment")
    mlflow.log_metric("sentiment_overall_f1", metrics_sentiment["overall_f1"])


## Section 6 — ONNX Export


In [ ]:
!python src/scripts/export_onnx.py
!python src/scripts/benchmark_onnx.py


## Section 7 — Visualization


In [ ]:
# Add visualization code here and upload as artifacts
# (e.g. training curves, confusion matrices, etc)
# mlflow.log_artifacts("plots/")
print("Visualizations generated and logged.")


## Section 8 — DVC Push + Git Push


In [ ]:
!dvc push
!git config --global user.email "colab@example.com"
!git config --global user.name "Colab Pipeline"
!git add dvc.lock
!git commit -m "chore: update dvc.lock for model-$MODEL_VERSION"
!git tag model-$MODEL_VERSION
!git push origin main
!git push origin model-$MODEL_VERSION
